# Notebook 01 — Tokens, probabilities, sampling, zero-shot and few-shot

This notebook makes several core LLM ideas concrete. Most cells use only Python/NumPy. The final tokenization section optionally uses Hugging Face Transformers.

Goals:
1. understand logits → softmax → probabilities;
2. see temperature change a distribution;
3. implement top-k and top-p sampling;
4. understand zero-shot vs few-shot as **prompt conditions**, not weight updates;
5. inspect real tokenizer output.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)


## 1. Logits → softmax

Imagine the model has only three possible next tokens. Logits are arbitrary real-valued scores. Softmax turns them into probabilities.


In [ ]:
tokens = np.array(["cat", "dog", "car"])
logits = np.array([2.0, 1.0, 0.0])

def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

probs = softmax(logits)
pd.DataFrame({"token": tokens, "logit": logits, "probability": probs})


### Exercise
Change the logits to `[2, 1.9, 0]`. What happens? Then try `[10, 1, 0]`.

**Research intuition:** probability behavior depends on relative logits, not simply which token ranks first.


## 2. Temperature

Temperature divides logits before softmax. Lower temperature sharpens; higher temperature flattens.


In [ ]:
def temperature_probs(logits, temperature):
    return softmax(logits / temperature)

rows = []
for T in [0.25, 0.5, 1.0, 2.0]:
    p = temperature_probs(logits, T)
    for token, prob in zip(tokens, p):
        rows.append({"temperature": T, "token": token, "probability": prob})

df = pd.DataFrame(rows)
df.pivot(index="temperature", columns="token", values="probability")


In [ ]:
for token in tokens:
    subset = df[df["token"] == token]
    plt.plot(subset["temperature"], subset["probability"], marker="o", label=token)
plt.xlabel("Temperature")
plt.ylabel("Probability")
plt.title("Temperature changes the next-token distribution")
plt.legend()
plt.show()


## 3. Sampling

Greedy decoding always selects the largest probability. Sampling draws from the distribution.

Run the cell repeatedly or increase `n` to see empirical frequencies approach the probabilities.


In [ ]:
p = temperature_probs(logits, 1.0)
n = 10_000
samples = np.random.choice(tokens, size=n, p=p)
pd.Series(samples).value_counts(normalize=True)


## 4. Top-k

Top-k retains only the k highest-probability tokens, renormalizes, and samples.


In [ ]:
def top_k_distribution(probs, k):
    keep = np.argsort(probs)[-k:]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

for k in [1, 2, 3]:
    print(k, dict(zip(tokens, top_k_distribution(p, k).round(3))))


## 5. Top-p / nucleus sampling

Top-p keeps the smallest highest-probability set whose cumulative probability reaches the threshold.


In [ ]:
def top_p_distribution(probs, threshold):
    order = np.argsort(probs)[::-1]
    sorted_p = probs[order]
    cumulative = np.cumsum(sorted_p)
    cutoff = np.searchsorted(cumulative, threshold, side="left")
    keep = order[:cutoff + 1]
    out = np.zeros_like(probs)
    out[keep] = probs[keep]
    return out / out.sum()

for threshold in [0.5, 0.8, 0.95, 1.0]:
    print(threshold, dict(zip(tokens, top_p_distribution(p, threshold).round(3))))


## 6. Zero-shot vs few-shot

The distinction is about demonstrations in the **context**.

### Zero-shot prompt

```
Classify sentiment as positive, negative, or neutral.
Text: The screen is beautiful but the battery is disappointing.
Label:
```

### Few-shot prompt

```
Classify sentiment as positive, negative, or neutral.

Text: I love the camera.
Label: positive

Text: The app crashes every hour.
Label: negative

Text: The package arrived Tuesday.
Label: neutral

Text: The screen is beautiful but the battery is disappointing.
Label:
```

No model weights changed. The demonstrations were simply included in context.

### Proper experiment

Do not test one prompt and declare victory. Create a held-out dataset, evaluate 0/1/3/5-shot conditions, repeat with different demonstration selections, and report both average score and variation.


In [ ]:
# A tiny scaffold for storing experimental conditions.
conditions = [
    {"name": "zero_shot", "n_examples": 0},
    {"name": "one_shot", "n_examples": 1},
    {"name": "three_shot", "n_examples": 3},
    {"name": "five_shot", "n_examples": 5},
]
pd.DataFrame(conditions)


## 7. Accuracy, precision, recall and F1

Use a tiny binary example to see why accuracy alone can be misleading.


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_true = [1,1,1,1,0,0,0,0,0,0]
y_pred = [1,1,0,0,0,0,0,0,0,0]

print("accuracy ", accuracy_score(y_true, y_pred))
print("precision", precision_score(y_true, y_pred))
print("recall   ", recall_score(y_true, y_pred))
print("f1       ", f1_score(y_true, y_pred))
print("confusion matrix\n", confusion_matrix(y_true, y_pred))


## 8. Optional: inspect a real tokenizer

This downloads a tokenizer the first time, so run it in an environment with internet access and `transformers` installed.


In [ ]:
# Uncomment when ready:
#
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained("gpt2")
#
# texts = [
#     "Machine learning is useful.",
#     "George Obaido",
#     "https://example.com/a?id=42",
#     "def softmax(x): return np.exp(x) / np.exp(x).sum()",
# ]
#
# for text in texts:
#     ids = tokenizer.encode(text)
#     pieces = tokenizer.convert_ids_to_tokens(ids)
#     print("\nTEXT:", text)
#     print("IDS:", ids)
#     print("TOKENS:", pieces)
#     print("COUNT:", len(ids))


## 9. Turn this notebook into a real study

Suggested study: **Prompting strategy × model size**

- Models: 2–4 open models at different parameter scales.
- Dataset: one public classification or QA dataset.
- Conditions: zero-shot, 1-shot, 3-shot, 5-shot.
- Repeat: 5 random demonstration selections.
- Metrics: accuracy/F1, token count, latency, variance.
- Analysis: Does few-shot help smaller models more? Is improvement worth added context cost? Which error categories change?

Save every raw model response. A research project is much easier to audit when you preserve outputs rather than only final scores.
